# TALENTIQ — WHAT PREPROCESSING AND FEATURE ENGINEERING ACTUALLY DID


### this notebook does not clean anything by itself
the real cleaning already happened inside src/preprocessing.py and src/feature_engineering.py when those files were run. here we are just loading what they produced and walking through it, so anyone reading this can follow what happened without going line by line through the source code.


## PART 1 — PREPROCESSING (src/preprocessing.py)


### -- what was wrong with the raw data
straight out of the CSV, the resume dataset had four problems we needed to deal with before any model could use it.

1. duplicate rows — same candidate showing up more than once, which would let the model just memorise that row instead of learning a general pattern
2. missing values — empty cells, and a model cannot work with an empty cell, it needs a number every time
3. outliers — a few candidates with unrealistic values, like years of experience that don't make sense, which can pull the model's decision boundary off in a bad direction
4. text columns — EducationLevel, CompanyType and the rest are words, not numbers, and models only work with numbers


### -- step 1, removed duplicate rows
any row that was an exact copy of another one was dropped, this one was straightforward, no real decision to make here.

### -- step 2, filled in the missing values
for number columns like CGPA and SkillsScore, the gaps were filled with the median of that column, median instead of mean because it does not get pulled around by outliers.
for category columns like EducationLevel, the gaps were filled with the mode, basically the most common value, which is the safest guess when there is nothing else to go on.

### -- step 3, capped the outliers using the IQR rule
for CGPA, YearsExperience, SkillsScore and SoftSkillsScore, we worked out a normal range using IQR, anything way outside the middle 50% of the data is treated as suspicious. instead of deleting these rows, the values were clipped down to the nearest acceptable boundary, so we keep the candidate but stop that one value from dragging the model around.

### -- step 4, turned the text categories into numbers
EducationLevel and UniversityTier actually have an order to them, High School comes before Bachelor, which comes before Master, then PhD. so these were ordinal encoded, basically mapped to 1, 2, 3, 4 in that order, which keeps the ranking meaning intact.
CompanyType and ProgrammingLanguages don't have any natural order, one category isn't "more" than another. so these were one-hot encoded instead, each category gets its own 0/1 column. this stops the model from assuming one category is mathematically bigger just because of how it was encoded.

### -- one important detail, the split happened before the encoding and scaling
the median values, the encoders and the scaler were all worked out using the training data only, then the exact same values were applied to the test data. if we had calculated these using the full dataset first, the test set would leak information into training, and our evaluation numbers later would look better than they would actually be in the real world.

### -- step 5, scaled the numbers
the numeric columns were on very different scales, CGPA goes up to 10, YearsExperience could go up to 30 or more. StandardScaler was used to bring every numeric column to the same scale, mean of 0 and standard deviation of 1, so no feature ends up dominating just because its raw numbers happen to be bigger.

### -- what got saved to disk and why
- artifacts/scaler.pkl — the exact scaling rule, so the same transformation can be applied to a brand new candidate later instead of guessing
- artifacts/encoder.pkl — same idea, the exact encoding rule
- data/splits/train.csv and data/splits/test.csv — the cleaned, encoded, scaled data, ready to be used for modelling


### -- let's actually look at the result


In [ ]:
import sys
sys.path.append('..')
import pandas as pd
import joblib

train_df = pd.read_csv("../data/splits/train.csv")
test_df  = pd.read_csv("../data/splits/test.csv")

print(f"Train shape: {train_df.shape}")
print(f"Test shape:  {test_df.shape}")
print()
print("First few rows of the cleaned, encoded, scaled training data:")
train_df.head()


In [ ]:
scaler = joblib.load("../artifacts/scaler.pkl")
encoders = joblib.load("../artifacts/encoder.pkl")

print("Numeric columns the scaler was fit on:")
print(list(scaler.feature_names_in_) if hasattr(scaler, "feature_names_in_") else "scaler loaded")

print()
print("Encoders saved:")
for key, info in encoders.items():
    print(f" - {key}: columns = {info['columns']}")


### -- this confirms the scaler and encoders were saved properly
we are just loading them back and printing what columns they were fit on, so we can confirm the train/serve symmetry actually holds, the same objects that transformed the training data are the ones that will transform a new candidate later.


## PART 2 — FEATURE ENGINEERING (src/feature_engineering.py)


### -- why we built new features instead of just using the raw columns
the raw columns describe a candidate, but they don't directly capture some bigger picture ideas that probably matter for hiring, like how employable someone is overall, or how complete their profile is. rather than hoping the model figures these patterns out by itself from the raw numbers, 7 new columns were hand built that bring in this domain knowledge directly.


### -- the 7 features, and the thinking behind each one

**EmployabilityScore** = 0.3×CGPA + 0.4×SkillsScore + 0.3×SoftSkillsScore
this is one combined score, weighted a bit more toward skills (0.4) than academics or soft skills (0.3 each), since skills usually matter more than GPA when it comes to hiring.

**PortfolioStrength** = Projects×0.5 + Hackathons×0.3 + ResearchPapers×0.2
measures hands-on proof of ability. projects get the highest weight since they are the most direct evidence of applied skill.

**TechnicalReadiness** = Languages×0.4 + Certifications×0.3 + SkillsScore×0.3
mixes breadth (how many languages someone knows) with depth (certifications and skill score) into one number.

**ExperienceQuality** = (YearsExperience×0.6 + Internships×0.4), then min-max normalized to 0–1
years of experience matters most here, but internships still count. normalizing to 0–1 makes it easier to compare against the other scores.

**LearningIndex** = Certifications + ResearchPapers + Hackathons
just a simple sum. all three point toward a habit of continuous learning, so there was no need to weight them differently.

**ProfileCompleteness** = percentage of a candidate's fields that are filled in and non-zero
a thin or incomplete profile might be its own signal, either inexperience or just low effort on the application.

**SkillExperienceGap** = abs(SkillsScore − YearsExperience×10)
flags a mismatch, like a high skill score with very little experience, or the other way round, which can be a useful signal on its own.

the last two (ProfileCompleteness and SkillExperienceGap) were added on top of the original five, specifically to catch patterns the first five didn't cover.

### -- why hand build these instead of letting the model find the patterns itself
simpler models like Logistic Regression can only combine the raw features in fairly basic ways. by pre-combining related columns into one meaningful score ahead of time, these patterns become much easier for every model to actually use, not just the more complex ones.


### -- let's actually look at the result


In [ ]:
cleaned_df = pd.read_csv("../data/processed/cleaned.csv")
feature_cols = joblib.load("../artifacts/feature_columns.pkl")

engineered = [
    "EmployabilityScore", "PortfolioStrength", "TechnicalReadiness",
    "ExperienceQuality", "LearningIndex", "ProfileCompleteness", "SkillExperienceGap"
]

print(f"Total feature columns saved for modeling: {len(feature_cols)}")
print()
print("The 7 engineered features, with summary stats:")
cleaned_df[engineered].describe()


In [ ]:
# quick sanity check, do these engineered features actually relate to Hired
target = "Hired"
if target in cleaned_df.columns:
    print("Average value of each engineered feature, split by Hired outcome:")
    cleaned_df.groupby(target)[engineered].mean()


### -- a quick sanity check on the new features
grouping by Hired and averaging each engineered feature lets us see straight away whether these scores actually move in a sensible direction between the two outcomes, a useful gut-check before trusting them in modelling.


## SUMMARY — WHAT TO TAKE AWAY

- preprocessing fixed the data quality problems — duplicates removed, missing values filled sensibly, outliers capped instead of deleted, text categories converted to numbers, everything scaled consistently, with train and test always kept separate so nothing leaks across
- feature engineering added domain knowledge on top — 7 new columns that pre-combine related raw signals into single, more meaningful scores, so the model doesn't have to discover these combinations on its own
- none of this was decided by EDA alone — EDA only pointed out that problems existed, for example "these 4 columns have outliers", the actual fix, the capping method, the formulas, were design decisions made inside these two files
- what this notebook does NOT do — it does not re-run preprocessing or feature engineering, it only loads what those scripts already produced and walks through it. if the source files change, run the pipeline again first, then come back and re-run this notebook to see the updated result
